# Mundial 2026 - Bloque 4

### Librerías

In [1]:
import os
import re
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

#### Primero leemos la clave

Revisamos si el fichero .env existe, en ese caso nos guardamos la API. En el caso contrario, lanza un error.

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()                      # lee el fichero .env si existe
except ImportError:
    print("python-dotenv no está instalado: se usará la variable de entorno del sistema.")

API_KEY = os.environ.get("API_FOOTBALL_KEY")

if API_KEY:
    print(f"Clave cargada")
else:
    print("No se ha encontrado API_FOOTBALL_KEY. Crea el fichero .env antes de seguir.")

Clave cargada


#### Creamos sesión con reintentos y constantes

La sesión creada utiliza: 
- La cabecera `x-apisports-key` indica nuestra API_KEY de la API de football
- `Retry` + `mount` marcan los límites ante errores posibles(429 y 5..)

In [ ]:
URL_BASE = "https://v3.football.api-sports.io"

sesion = requests.Session()
sesion.headers.update({"x-apisports-key": API_KEY or ""})
reintentos = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])  # (veces, tiempo, errores ante los que actua)
sesion.mount("https://", HTTPAdapter(max_retries=reintentos))

# Identificadores de competiciones en API-Football
ID_MUNDIAL = 1
TEMPORADA_MUNDIAL = 2026
TEMPORADA_CLUBES = 2025            # temporada 2025/26: los clubes en los que jugaban durante el Mundial
LIGAS = {
    "Premier League": 39,
    "LaLiga": 140,
    "Bundesliga": 78,
    "Serie A": 135,
    "Ligue 1": 61}

PAUSA = 7                          # segundos entre llamadas: el plan gratuito limita también las llamadas por minuto
CARPETA_DATOS = Path("cache_api")
CARPETA_DATOS.mkdir(exist_ok=True)


def buscar_fichero(nombre):
    """
    Busca un fichero de datos del proyecto en la carpeta actual, en la superior o en datos/
    """
    for carpeta in [Path("."), Path(".."), Path("datos"), Path("../datos")]:
        if (carpeta / nombre).exists():
            return carpeta / nombre
    raise FileNotFoundError(f"No se encuentra {nombre}. Copia los Excel del Bloque 1 junto al notebook.")

#### Función para llamar a la API y función para paginación

La primera función *api_get()* hace lo siguiente: 
- Primero comprueba que los datos hayan sido descargados. En el caso de que si, los devuelve.
- Envía una petición a la API y comprueba si ha habido un error.
- Guardamos los datos en un  fichero dentro del directorio creado en el apartado anterior

La segunda función pide la información de todas las páginas

In [ ]:
def api_get(endpoint, params):
    # Nombre del fichero donde se guarda esta respuesta, p. ej. "players_1_2026_5.json"
    nombre = endpoint.replace("/", "_") + "_" + "_".join(str(v) for v in params.values()) + ".json"
    ruta = CARPETA_DATOS / nombre

    # Si ya la teníamos guardada, la leemos del disco (no gasta llamadas)
    if ruta.exists():
        with open(ruta, "r", encoding="utf-8") as archivo:
            return json.load(archivo)

    # Si no, la pedimos a la API
    r = sesion.get(f"{URL_BASE}/{endpoint}", params=params, timeout=10)
    r.raise_for_status()
    datos = r.json()

    # API-Football responde 200 aunque haya un error: hay que mirar el campo "errors"
    if datos["errors"]:
        raise RuntimeError(datos["errors"])

    # Guardamos la respuesta y esperamos antes de la siguiente llamada
    with open(ruta, "w", encoding="utf-8") as archivo:
        json.dump(datos, archivo)
    print("Llamada a la API:", endpoint, params)
    time.sleep(PAUSA)
    return datos


def api_get_paginas(endpoint, params):
    # Pide todas las páginas de un endpoint y junta los resultados en una lista
    registros = []
    pagina = 1
    total_paginas = 1
    while pagina <= total_paginas:
        params_pagina = params.copy()
        params_pagina["page"] = pagina
        datos = api_get(endpoint, params_pagina)
        registros.extend(datos["response"])
        total_paginas = datos["paging"]["total"]
        pagina += 1
    return registros

## Ejercicio 2) Analizar la documentación de API_Football

Antes de programar leemos la documentación ([api-football.com/documentation-v3](https://www.api-football.com/documentation-v3)):

| Pregunta | Respuesta en API-Football |
|---|---|
| URL | `https://v3.football.api-sports.io` |
| Devuelve | JSON con `response` (los datos), `results`, `paging` y `errors` |
| + resultados | Paginación con `page` y `paging.total` |

### Endpoints que vamos a usar

| Endpoint | Parámetros | Qué devuelve | Uso |
|---|---|---|---|
| `/status` | — | Plan contratado y llamadas consumidas hoy | Control del límite |
| `/leagues` | `id` | Temporadas disponibles de una competición y su cobertura | Comprobar que existe el Mundial 2026 |
| `/players` | `league`, `season`, `page` | Jugadores con **fecha de nacimiento** (`birth.date`) y estadísticas; **20 por página** | Ejercicio 3 |
| `/teams` | `league`, `season` | Equipos de una liga | Ejercicio 4 |
| `/players/squads` | `team` | **Plantilla completa** de un equipo en una sola llamada | Ejercicio 4 |

#### 2.1 Vemos el estado de nuestra cuenta con */status*.

In [ ]:
r = sesion.get(f"{URL_BASE}/status", timeout=10)
r.raise_for_status()
estado = r.json()["response"]

print("Plan:              ", estado["subscription"]["plan"])
print("Llamadas hoy:      ", estado["requests"]["current"], "de", estado["requests"]["limit_day"])
print("Llamadas en caché: ", len(list(CARPETA_DATOS.glob("*.json"))), "respuestas guardadas")

#### 2.2 Comprobamos si está disponible el Mundial 2026 en esta API con */leagues*.

In [ ]:
datos_liga = api_get("leagues", {"id": ID_MUNDIAL})
mundial_info = datos_liga["response"][0]

temporadas = pd.json_normalize(mundial_info["seasons"])
print("Competición:", mundial_info["league"]["name"])
display(temporadas[["year", "start", "end", "coverage.players"]].tail())

assert TEMPORADA_MUNDIAL in temporadas["year"].values, "La temporada 2026 no aparece en la API"

## Ejercicio 3) ¿Influye el mes de nacimiento en el éxito deportivo?

En el fútbol los jugadores se agrupan por año de nacimiento, y un niño nacido en enero es casi un año mayor que aquellos nacidos en diciembre. Esto significa que en los primeros años de fútbol de su vida tienen más fuerza por lo que destacan más. Veremos si esto es cierto.